# 🧠 Pillar B: Yuan-Yuan's AI Knowledge Base Assistant
**Project:** Yuan-Yuan's AI Knowledge Base  
**System Role:** Retrieval-Augmented Generation (RAG) Interface & Evaluator

### 🎯 Objective
This is the user-facing interface. It connects the persistent ChromaDB vector store to Azure OpenAI to provide accurate, cited answers based strictly on Yuan-Yuan's curated knowledge base.

### 🛠️ Key Architectural Features:
1. **Semantic Retrieval:** Uses vector similarity to find relevant context, allowing the assistant to understand the "intent" behind a question rather than just keywords.
2. **Hallucination Guardrails:** Implements strict system prompts that require the AI to cite sources and admit ignorance if the information is missing from the context.
3. **Low-Latency Architecture:** Decouples heavy data processing (Pillar A) from the user interface for near-instant response times.
4. **BERT Semantic Evaluation:** Uses a local transformer model to calculate the similarity between the question and the response, providing a quantitative "Confidence Score."

---
*Status: Ready for Querying. Operates independently of the Confluence API.*

In [3]:
# --- STEP 1: ENVIRONMENT & UTILITIES ---
# This cell must handle the "Environment Handshake" and ensure SQLite is upgraded and rag_util is loaded correctly. 
import pysqlite3
import sys
import os

# Essential SQLite override for ChromaDB compatibility
sys.modules["sqlite3"] = pysqlite3

import chromadb
from openai import AzureOpenAI
import rag_util  # This loads your model and cleaning logic

from sentence_transformers import util  # For answer quality verification
# Initialize Azure Client
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

print("✅ Environment Ready: SQLite Overridden, Azure Connected, rag_util Loaded.")

✅ Environment Ready: SQLite Overridden, Azure Connected, rag_util Loaded.


In [ ]:
#%pip install tiktoken  # Ensure tiktoken is available for token counting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.6 MB/s  0:00:00

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# --- STEP 1.0: SYSTEM HEALTH CHECK ---
def run_health_check():
    try:
        # Connect to the persistent store
        db_client = chromadb.PersistentClient(path="./chroma_data")
        collection = db_client.get_collection(name="document_store")
        
        # 1. Count Total Chunks
        total_chunks = collection.count()
        
        # 2. Count Unique Pages (extracted from metadata)
        all_data = collection.get(include=['metadatas'])
        unique_pages = set(m['source_title'] for m in all_data['metadatas'])
        total_pages = len(unique_pages)
        
        print("📊 SYSTEM STATUS: ONLINE")
        print("-" * 25)
        print(f"✅ Knowledge Base: Linked (./chroma_data)")
        print(f"📑 Curated Pages:  {total_pages}")
        print(f"🧩 Searchable Chunks: {total_chunks}")
        print("-" * 25)
        
    except Exception as e:
        print(f"⚠️ Health Check Failed: {e}")

run_health_check()

⚠️ Health Check Failed: Collection [document_store] does not exist


In [5]:
# --- STEP 1.1: KNOWLEDGE BASE AUDIT ---
def audit_database():
    db_client = chromadb.PersistentClient(path="./chroma_data")
    collection = db_client.get_collection(name="document_store")
    
    # Get all metadata
    all_data = collection.get(include=['metadatas'])
    metadatas = all_data['metadatas']
    
    # Count chunks per source
    stats = {}
    for m in metadatas:
        title = m['source_title']
        stats[title] = stats.get(title, 0) + 1 # returns the current count or 0 if new, then adds 1
    
    print("📊 KNOWLEDGE BASE BREAKDOWN")
    print("-" * 30)
    for title, count in sorted(stats.items(), key=lambda x: x[1], reverse=True):
        print(f"📄 {title.ljust(40)} | {count} chunks")
    print("-" * 30)
    print(f"Total: {len(stats)} Pages | {sum(stats.values())} Chunks")

audit_database()

📊 KNOWLEDGE BASE BREAKDOWN
------------------------------
📄 Python Set Up Guide                      | 34 chunks
📄 Connect and Aim Ahead Home Page          | 4 chunks
📄 Weekly Update Process                    | 4 chunks
📄 Work with SAS Datasets and Programs without a PC SAS Installation | 4 chunks
📄 Working with Large SAS Datasets          | 3 chunks
📄 PC SAS Retirement Home                   | 2 chunks
📄 DailyBio                                 | 1 chunks
📄 Python Virtual Environment Set Up:       | 1 chunks
📄 Connecting Power BI to SAS Datasets      | 1 chunks
------------------------------
Total: 9 Pages | 54 Chunks


In [ ]:
# --- STEP 2: THE ENGINE FUNCTIONS ---
# These functions are workhorse functions. They handle retrieval, prompt formatting, and LLM interaction.
def get_context_for_llm(question, n_results=3):
    """Retrieves 3 relevant chunks from your 56-chunk collection."""
    db_client = chromadb.PersistentClient(path="./chroma_data")
    collection = db_client.get_collection(name="document_store")
    
    # Use the model embedded in rag_util
    query_vector = rag_util.model.encode([question])[0].tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=n_results)# this wont embed the query text, because the query has already converted to a vector using the local model. 
        
    context_output = ""
    for i in range(len(results['documents'][0])):
        source = results['metadatas'][0][i]['source_title']
        content = results['documents'][0][i]
        context_output += f"\n--- FROM DOCUMENT: {source} ---\n{content}\n"
    return context_output

def get_budgeted_context(query, collection, max_tokens=2000):
    results = collection.query(query_texts=[query], n_results=10)
    
    # 1. Create a dictionary to group content by source
    grouped_data = {} # Format: {"Title": ["chunk1", "chunk2"]}
    current_tokens = 0
    
    for i in range(len(results['documents'][0])):
        chunk_text = results['documents'][0][i]
        metadata = results['metadatas'][0][i]
        title = metadata['source_title']
        
        chunk_tokens = rag_util.count_tokens(chunk_text)
        
        if (current_tokens + chunk_tokens) <= max_tokens:
            # If title is new, start a list; otherwise append
            if title not in grouped_data:
                grouped_data[title] = []
            grouped_data[title].append(chunk_text)
            current_tokens += chunk_tokens
        else:
            print(f"⚠️ Budget reached. Skipping chunk {i+1} to stay under {max_tokens} tokens.")
            break
            
    # 2. Build the final string grouped by document
    formatted_context = ""
    for title, chunks in grouped_data.items():
        formatted_context += f"\n=== DOCUMENT: {title} ===\n"
        # Join chunks from the same document with a simple newline
        formatted_context += "\n".join(chunks) + "\n"
        
    return formatted_context

def generate_data_ops_response(question, context):
    """Formats the prompt for Yuan-Yuan's Assistant."""
    return f"""
    You are a Yuan-Yuan's confluence resource AI assistant. 
    Use the following retrieved context to answer the user's question.
    
    RULES:
    1. You are a technical expert. Synthesize the provided context into a cohesive, step-by-step guide. If the context describes two different tasks (e.g., Python setup and DailyBio fixes), create a clear transition between them.
    2. If the answer is not in the context, say: "I'm sorry, I don't have documentation on that in my current knowledge base."
    3. Be specific: If a step-by-step process (like DailyBio or Weekly Update) is mentioned, list it clearly.
    4. CITATION: Always mention which Confluence page the information came from (e.g., "According to the 'Python Set Up Guide'...").

    CONTEXT:
    {context}

    USER QUESTION: {question}
    
    YUAN-YUAN DATA OPS ADVICE:
    """


def get_answer_confidence(question, ai_answer):
    """
    BERT Semantic Evaluation:
    Calculates how well the answer matches the user's intent.
    """
    # 1. Convert question and answer into vectors using your local model
    q_vector = rag_util.model.encode(question, convert_to_tensor=True)
    a_vector = rag_util.model.encode(ai_answer, convert_to_tensor=True)
    
    # 2. Compute Cosine Similarity (Math-based semantic match)
    cosine_scores = util.cos_sim(q_vector, a_vector)
    score = cosine_scores.item()
    
    # 3. Assign a label
    if score > 0.75:
        label = "✅ High Confidence"
    elif score > 0.50:
        label = "⚠️ Moderate Confidence"
    else:
        label = "❌ Low Confidence / Verify Source"
        
    return score, label
def ask_yuan_yuan_safe(question):
    try:
        # Check if environment variables are actually loaded
        if not os.getenv("AZURE_OPENAI_KEY"):
            return "❌ Error: API Key not found. Please check your environment variables."

        # 1. Access DB
        db_client = chromadb.PersistentClient(path="./chroma_data")
        collection = db_client.get_collection(name="document_store")

        # 2. RETRIEVE (using the new Budgeted Context)
        context = get_budgeted_context(question, collection, max_tokens=2500)
        
        if not context.strip():
            return "🔍 I searched the knowledge base but couldn't find relevant documentation."

        # 3. GENERATE (using the General Data Ops Prompt)
        prompt = generate_data_ops_response(question, context)
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        
        ai_answer = response.choices[0].message.content
        
        # 4. EVALUATE (Keep your BERT similarity check)
        score, label = get_answer_confidence(question, ai_answer)
        
        return f"{ai_answer}\n\n📊 EVALUATION: {score:.2f} ({label})"

    except Exception as e:
        return f"⚠️ System Note: Error connecting to AI brain: {str(e)}"

In [ ]:
# --- STEP 3: INTERFACE ---
# This is where you can ask questions and get answers. The function `ask_yuan_yuan` will handle the entire retrieval and response generation process.
user_query = "I'm setting up Python and need to fix the DailyBio file, how do I do both?"

answer = ask_yuan_yuan_safe(user_query)
print(answer)

To set up Visual Studio Code for SAS, you need to install the following extension:

1. **SAS Extension**: This extension is published by SAS Institute Inc. It provides functionality for SAS code editing that is similar to the PC SAS code editor. 

   - To install it, open Visual Studio Code, go to the Extensions panel, search for the SAS extension, and install it. If you want the editor to resemble the traditional SAS editor, set the color theme to SAS Light.

Additionally, for enhanced functionality, you may consider installing the **Log File Highlighter** extension, which provides syntax highlighting for SAS logs.

According to the "Work with SAS Datasets and Programs without a PC SAS Installation" guide, these extensions will help you effectively edit and manage SAS code within Visual Studio Code.

📊 EVALUATION: 0.66 (⚠️ Moderate Confidence)
